In [1]:
import rasterio
import xarray as xr
import glob
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from scipy.ndimage import median_filter
from rasterio.io import MemoryFile
from rasterio.features import shapes
from shapely.geometry import shape
#from skimage.morphology import (
#    binary_opening, binary_closing, binary_erosion, binary_dilation,
#    disk, remove_small_objects, remove_small_holes
#)
#from scipy.ndimage import binary_fill_holes

import skimage as ski
import numpy as np

In [5]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/morphFilter")
output_folder.mkdir(exist_ok=True)

input_folder_s2 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics")
input_folder_s1 = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2017")

In [3]:
def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [15]:
# find corresponding coherence file
#s1_coherence_files = find_files(input_folder_s1 , "sar_coherence_composite_all_swaths_thresh_0.3_masked.tiff")
#print(f"Found {len(s1_coherence_files)} S1 coherence files.")

# find acending and descending coherence files
s1_coherence_files_asc = find_files(input_folder_s1 , "masked_ascending_0.5_morphFilter_3.tif")
print(f"Found {len(s1_coherence_files_asc)} S1 coherence files (ascending).")

s1_coherence_files_desc = find_files(input_folder_s1 , "masked_descending_0.5_morphFilter_3.tif")
print(f"Found {len(s1_coherence_files_desc)} S1 coherence files (descending).") 

# find corresponding NDSI file
s2_ndsi_files = find_files(Path(input_folder_s2 / "ndsi" / "2017"), "ndsi_composite_*.tiff")
print(f"Found {len(s2_ndsi_files)} S2 NDSI files.")

# find corresponding ratio file
s2_ratio_files = find_files(Path(input_folder_s2 / "ratio" / "2017"), "ratio_composite_*.tiff")
print(f"Found {len(s2_ratio_files)} S2 ratio files.")

with rasterio.open(s1_coherence_files_asc[0]) as src:
    s1_asc = src.read(1)
    s1_asc_meta = src.meta
    print("S1 coherence (ascending):", s1_asc_meta['crs'], s1_asc_meta['transform'], s1_asc.shape)

with rasterio.open(s1_coherence_files_desc[0]) as src:
    s1_desc = src.read(1)
    s1_desc_meta = src.meta
    print("S1 coherence (descending):", s1_desc_meta['crs'], s1_desc_meta['transform'], s1_desc.shape)

with rasterio.open(s2_ndsi_files[0]) as src:
    s2_ndsi_image = src.read(1)
    s2_meta = src.meta
    print("S2 NDSI:", s2_meta['crs'], s2_meta['transform'], s2_ndsi_image.shape)

with rasterio.open(s2_ratio_files[0]) as src:
    s2_ratio_image = src.read(1)
    s2_ratio_meta = src.meta
    print("S2 ratio:", s2_ratio_meta['crs'], s2_ratio_meta['transform'], s2_ratio_image.shape)

Found 1 S1 coherence files (ascending).
Found 1 S1 coherence files (descending).
Found 1 S2 NDSI files.
Found 1 S2 ratio files.
S1 coherence (ascending): EPSG:32632 | 10.00, 0.00, 608290.00|
| 0.00,-10.00, 5218915.70|
| 0.00, 0.00, 1.00| (7422, 13777)
S1 coherence (descending): EPSG:32632 | 10.00, 0.00, 608290.00|
| 0.00,-10.00, 5218920.00|
| 0.00, 0.00, 1.00| (7422, 13777)


/tmp/ipykernel_1646943/3142787845.py:21: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s1_asc = src.read(1)
/tmp/ipykernel_1646943/3142787845.py:26: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s1_desc = src.read(1)


S2 NDSI: EPSG:32632 | 10.00, 0.00, 610950.00|
| 0.00,-10.00, 5218920.00|
| 0.00, 0.00, 1.00| (7422, 13511)
S2 ratio: EPSG:32632 | 10.00, 0.00, 610950.00|
| 0.00,-10.00, 5218920.00|
| 0.00, 0.00, 1.00| (7422, 13511)


/tmp/ipykernel_1646943/3142787845.py:31: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s2_ndsi_image = src.read(1)
/tmp/ipykernel_1646943/3142787845.py:36: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s2_ratio_image = src.read(1)


In [14]:
def align_to_reference(src_array, src_transform, src_crs, ref_meta, resampling=Resampling.nearest):
    #Resample src_array onto the reference grid defined by ref_meta.
    aligned = np.zeros((ref_meta['height'], ref_meta['width']), dtype=src_array.dtype)
    reproject(
        source=src_array,
        destination=aligned,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=ref_meta['transform'],
        dst_crs=ref_meta['crs'],
        resampling=resampling
    )
    return aligned

In [16]:
ref_meta = s1_asc_meta  

s2_ndsi_aligned = align_to_reference(
    s2_ndsi_image, s2_meta['transform'], s2_meta['crs'], ref_meta,
    resampling=Resampling.nearest
)

s2_ratio_aligned = align_to_reference(
    s2_ratio_image, s2_ratio_meta['transform'], s2_ratio_meta['crs'], ref_meta,
    resampling=Resampling.nearest
)

s1_asc_aligned = align_to_reference(
    s1_asc, s1_asc_meta['transform'], s1_asc_meta['crs'], ref_meta, resampling= Resampling.nearest
)

s1_desc_aligned = align_to_reference(
    s1_desc, s1_desc_meta['transform'], s1_desc_meta['crs'], ref_meta, resampling= Resampling.nearest
)

s2_ndsi_mask = s2_ndsi_aligned.astype(bool)
s2_ratio_mask = s2_ratio_aligned.astype(bool)
s1_asc = s1_asc_aligned.astype(bool)
s1_desc = s1_desc_aligned.astype(bool)

combined = s2_ndsi_mask | s2_ratio_mask | s1_asc | s1_desc

#filtered_raster = median_filter(combined, size=3)

out_meta = ref_meta.copy()
out_meta.update({
    "count": 1,
    "nodata": 0
})

with rasterio.open(output_folder / "s1_s2_asc_desc_combined_2017.tiff", "w", **out_meta) as dest:
    dest.write(combined, 1) 

In [17]:
struct = ski.morphology.disk(3)  

# speckle / isolated false-positive pixels
opened = ski.morphology.opening(combined, struct)

# bridge small gaps 
closed = ski.morphology.closing(opened, struct)

with rasterio.open(output_folder / "s1_s2_asc_desc_morphFilter_3_2017.tiff", "w", **out_meta) as dest:
    dest.write(opened, 1) 

# dilation
#dilated = ski.morphology.dilation(closed, struct)
#with rasterio.open(output_folder / "s1_s2_composite_2023_morphFilter_2_dilated.tiff", "w", **out_meta) as dest:
#    dest.write(dilated, 1) 

In [25]:
s1_coherence_files = find_files(input_folder_s1 , "coh_max_mosaic_thresh_0.3_masked.tiff")
print(f"Found {len(s1_coherence_files)} S1 coherence files.")

with rasterio.open(s1_coherence_files[0]) as src:
    s1_img = src.read(1)
    s1_meta = src.meta
    print("S1 coherence (descending):", s1_meta['crs'], s1_meta['transform'], s1_img.shape)

Found 1 S1 coherence files.
S1 coherence (descending): PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",9],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] | 10.00, 0.00, 608290.00|
| 0.00,-10.00, 5218915.68|
| 0.00, 0.00, 1.00| (7422, 13777)


In [12]:
struct = ski.morphology.disk(3)  

combined = s2_ndsi_mask | s2_ratio_mask | s1_img

# speckle / isolated false-positive pixels
opened = ski.morphology.opening(combined, struct)

# bridge small gaps 
closed = ski.morphology.closing(opened, struct)

with rasterio.open(output_folder / "s1_s2_mosaic_morphFilter_2017_3.tiff", "w", **out_meta) as dest:
    dest.write(opened, 1) 


NameError: name 's1_img' is not defined

In [23]:
results = (
    {'geometry': shape(geom), 'value': val}
    for geom, val in shapes(closed.astype(np.uint8), transform=ref_meta['transform'])
    if val == 1
)

gdf = gpd.GeoDataFrame.from_records(results)
gdf = gdf.set_geometry('geometry')
gdf = gdf.set_crs(ref_meta['crs'])

# save to disk
output_path = output_folder / "glacier_outlines_2023_v1.shp"  
gdf.to_file(output_path)
print(f"Saved {len(gdf)} polygons to {output_path}")

Saved 1568 polygons to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/morphFilter/glacier_outlines_2023_v1.shp
